# Lab 6A: Text-to-Speech with SpeechT5
### AIML ZG527: Audio Analysis
**Authored by:** Seetha Parameswaran, BITS Pilani WILP


### Learning Objectives
- Synthesize speech using SpeechT5 (Transformer TTS) + HiFi-GAN vocoder
- Visualize mel-spectrogram of generated audio
- Normalize numbers/currency/dates before synthesis
---
### Setup and Import Libraries
```
!pip install transformers datasets soundfile num2words --break-system-packages
``` 

In [ ]:
!pip install -q transformers soundfile --break-system-packages
!pip install -q num2words --break-system-packages
!pip install -q datasets --break-system-packages

In [ ]:
import IPython.display as ipd
import librosa, librosa.display
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
from num2words import num2words
import re

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import torch

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

## Text to speech of text only

In [ ]:
TEXT = "The quick brown fox jumps over the lazy dog."

inputs = processor(text=TEXT, return_tensors="pt")

speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)


ipd.display(ipd.Audio(speech.numpy(), rate=16000))

In [ ]:
# Visualize mel-spectrogram of generated audio (acoustic model output analogue)

mel = librosa.feature.melspectrogram(y=speech.numpy(), sr=16000, n_mels=80)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(10,3))
librosa.display.specshow(mel_db, sr=16000, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title("Mel-Spectrogram of Synthesized Speech")
plt.tight_layout()
plt.show()

## TTS for text with numbers

In [ ]:

def normalize_text(text):
    text = re.sub(r'\$(\d+\.?\d*)', lambda m: num2words(float(m.group(1))) + " dollars", text)
    text = re.sub(r'(\d+)/(\d+)', lambda m: f"{num2words(int(m.group(1)), to='ordinal')} of {num2words(int(m.group(2)))}", text)
    text = re.sub(r'\d+', lambda m: num2words(int(m.group())), text)
    return text

TEXT_NUMBERS = normalize_text("I have 3 apples and paid $5.50 on 12/25.")
print(TEXT_NUMBERS)

In [ ]:
# Text normalization demo: numbers vs plain text

inputs = processor(text=TEXT_NUMBERS, return_tensors="pt")
speech2 = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

ipd.display(ipd.Audio(speech2.numpy(), rate=16000))

print("Notice how the number/currency/date normalization step handles this internally.")

## Summary

In this lab, you learned:

1. **SpeechT5 TTS**: Transformer-based acoustic model + HiFi-GAN vocoder
2. **Speaker Embeddings**: x-vector conditioning enables different voice characters
3. **Text Normalization**: Numbers/currency/dates must be pre-normalized — SpeechT5 has no built-in normalizer
4. **Mel-Spectrogram**: Visual representation of synthesized speech

### Key Takeaways
- Unlike Tacotron2, SpeechT5 requires manual text normalization (num2words) before synthesis
- Speaker embedding conditioning is the same mechanism used later in voice cloning
- Lightweight `transformers`-only install avoids heavy multilingual dependency chains